# Тесты: интервальная математика, чёрные ящики, стохастические методы (Лаба 2)

In [1]:
import math
import src.ConstructiveNumber as CNum
import src.IntervalMath as IM
import src.BlackBox2 as BB2
from src.StochasticOptimizers import simulated_annealing, genetic_algorithm


## Шаг 1: интервальные версии cos/sin/exp/sqrt/round

In [2]:
# cos: отрезок без экстремума внутри -- просто min/max границ
r = IM.cn_cos(CNum.ConstructiveNumber(0, 1))
assert abs(float(r.a) - math.cos(1)) < 1e-6 and abs(float(r.b) - 1.0) < 1e-6

# cos: отрезок, содержащий максимум (0 внутри -> 2*pi*0=0 попадает)
r = IM.cn_cos(CNum.ConstructiveNumber(-1, 1))
assert float(r.b) == 1.0

# cos: отрезок вокруг pi -- содержит минимум -1
r = IM.cn_cos(CNum.ConstructiveNumber(3, 3.3))
assert float(r.a) == -1.0

# sin: отрезок вокруг pi/2 -- содержит максимум 1
r = IM.cn_sin(CNum.ConstructiveNumber(1.5, 1.7))
assert abs(float(r.b) - 1.0) < 1e-3

# exp монотонный
r = IM.cn_exp(CNum.ConstructiveNumber(0, 1))
assert abs(float(r.a) - 1.0) < 1e-6 and abs(float(r.b) - math.e) < 1e-3

# sqrt монотонный
r = IM.cn_sqrt(CNum.ConstructiveNumber(4, 9))
assert float(r.a) == 2.0 and float(r.b) == 3.0

# round -- консервативная (широкая) оценка на разрыве
r = IM.cn_round(CNum.ConstructiveNumber(0.4, 0.6))
assert r.a <= 0 and r.b >= 1


## Шаг 2: чёрные ящики -- Rastrigin-2, Ackley-2, Desmos

In [3]:
# Rastrigin-2: минимум f(0,0)=0
assert abs(BB2.Rastrigin2.func([0.0, 0.0])) < 1e-9

# Ackley-2: минимум f(0,0)=0
assert abs(BB2.Ackley2.func([0.0, 0.0])) < 1e-9

# Desmos: сверка с ручным расчётом
v = BB2.Desmos.func([0.0, 0.0])
assert abs(v - 149.0) < 1e-6
v2 = BB2.Desmos.func([1.0, 1.0])
assert abs(v2 - 73.0) < 1e-6


In [4]:
# Совместимость с ConstructiveNumber: вырожденный интервал = обычное число
x_cn = [CNum.ConstructiveNumber(1, 1), CNum.ConstructiveNumber(1, 1)]
v_cn = BB2.Ackley2.func(x_cn)
v_float = BB2.Ackley2.func([1.0, 1.0])
assert abs(float(v_cn.a) - v_float) < 1e-3

# На широком интервале получаем интервал на выходе (не падает с ошибкой)
x_wide = [CNum.ConstructiveNumber(-0.1, 0.1)] * 2
_ = BB2.Rastrigin2.func(x_wide)
_ = BB2.Desmos.func(x_wide)


## Шаг 3: имитация отжига (Simulated Annealing)

In [5]:
res = simulated_annealing(BB2.Rastrigin2, x0=[4.5, -3.7], delta=0.8, gamma=0.998, max_iter=5000, seed=1)
print('SA Rastrigin2: f_min =', res['f_min'], ' func_calls =', res['func_calls'])
assert res['func_calls'] == 5001  # 1 на старте + по 1 на каждую итерацию


SA Rastrigin2: f_min = 0.022444136820478633  func_calls = 5001


In [6]:
res = simulated_annealing(BB2.Ackley2, x0=[4.0, -3.5], delta=0.5, gamma=0.998, max_iter=5000, seed=1)
print('SA Ackley2:    f_min =', res['f_min'])

res = simulated_annealing(BB2.Desmos, x0=[2.5, -2.0], delta=0.3, gamma=0.998, max_iter=5000, seed=1)
print('SA Desmos:     f_min =', res['f_min'])


SA Ackley2:    f_min = 0.02258826852376572
SA Desmos:     f_min = 26.59523239723063


## Шаг 4: генетический алгоритм

In [7]:
res = genetic_algorithm(BB2.Rastrigin2, bounds=[(-5.12,5.12)]*2, pop_size=40, n_parents=20, max_iter=200, seed=42)
print('GA Rastrigin2: f_min =', res['f_min'], ' func_calls =', res['func_calls'])
assert res['f_min'] < 0.01


GA Rastrigin2: f_min = 2.8269516144874274e-05  func_calls = 8040


In [8]:
res = genetic_algorithm(BB2.Ackley2, bounds=[(-5,5)]*2, pop_size=40, n_parents=20, max_iter=200, seed=42)
print('GA Ackley2:    f_min =', res['f_min'])
assert res['f_min'] < 0.01

res = genetic_algorithm(BB2.Desmos, bounds=[(-3,3)]*2, pop_size=40, n_parents=20, max_iter=200, seed=42)
print('GA Desmos:     f_min =', res['f_min'])
assert res['f_min'] < 0.01


GA Ackley2:    f_min = 0.00035368729985307823
GA Desmos:     f_min = 0.00026587071010008466
